# 03 — Analysis & Visualization

Load all scratch experiment results and compare loss, VRAM, throughput, latency, and instruction-following outputs.

In [ ]:
from pathlib import Path
import sys
import torch

PROJECT_ROOT = Path("/content/qLoRA").resolve()
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
import subprocess

subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "origin", "main"], check=False)


In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "requirements.txt")])


## 1. Load Results

In [ ]:
from qlora_scratch.analysis import (
    build_instruction_tuning_table,
    build_method_comparison_table,
    load_all_metrics,
    metrics_to_frame,
    plot_lora_rank_ablation,
    plot_throughput_comparison,
    plot_system_metrics,
    plot_training_curve,
    plot_training_loss,
    plot_training_time_comparison,
    plot_vram_comparison,
    results_summary_table,
)

results = load_all_metrics(RESULTS_DIR)
summary = results_summary_table(results)
print(f"Loaded {len(results)} experiment results")
summary


## 2. Summary Table

In [ ]:
display_cols = [
    "experiment",
    "model",
    "lora_rank",
    "eval_loss",
    "perplexity",
    "peak_vram_mb",
    "peak_reserved_vram_mb",
    "tokens_per_second",
    "avg_generation_latency_s",
    "avg_generation_tokens_per_second",
    "wall_time_s",
]
summary[display_cols]


## 3. LoRA vs QLoRA Comparison

In [ ]:
comparison = build_method_comparison_table(summary)
comparison


## 4. Training Loss Curves

In [ ]:
fig = plot_training_loss(results, title="Scratch QLoRA Training Loss — All Runs")
fig


## 5. VRAM Usage Comparison

In [ ]:
fig = plot_vram_comparison(summary)
fig


## 6. Throughput Comparison

In [ ]:
fig = plot_throughput_comparison(summary)
fig


## 7. Training Time Comparison

In [ ]:
fig = plot_training_time_comparison(summary)
fig


## 8. LoRA Rank Ablation

In [ ]:
fig = plot_lora_rank_ablation(results)
fig


## 9. Detailed System Metrics For One Run

In [ ]:
metrics = results[0]
metrics_to_frame(metrics)


In [ ]:
fig = plot_system_metrics(metrics)
fig


## 10. Pre/Post Instruction-Tuning Comparison

In [ ]:
instruction_table = build_instruction_tuning_table(metrics)
instruction_table


In [ ]:
for row in instruction_table.to_dict(orient="records"):
    print(f"=== PROMPT {row['prompt_id']} ===")
    print(row["prompt"])
    print("\n--- PRE-FINETUNE ---")
    print(row["pre_response"])
    print("\n--- POST-FINETUNE ---")
    print(row["post_response"])
    print("\nLatency (s):", row["pre_latency_s"], "->", row["post_latency_s"])
    print("Tokens/sec:", row["pre_tokens_per_second"], "->", row["post_tokens_per_second"])
    print("\n")
